In [27]:
import numpy as np
from PIL import Image
import tensorflow as tf
import matplotlib.pyplot as plt
import warnings

# 경고를 보이지 않도록 설정해요
warnings.filterwarnings('ignore')

# 필요한모듈 import
import os
import glob # 조건에 맞는 파일명을 찾아서 싹 가져오는 함(이미지 찾기)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.utils import plot_model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Concatenate, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

** TFRecord 만들기 **
1. 저장할 데이터를 먼저 준비해요!
2. 각 데이터를 Feature로 변환하고 이를 모은 Features를 생성해요!
3. Features를 이용해서 Example을 생성해요!
4. TFRecordWriter를 이용해서 Example객체를 TFRecord파일에 저장!
-> 해당폴더의 전체 이미지 파일을 TFRecord로 저장해보고
   저장된 TFRecord에서 이미지 정보를 추출해 정상적으로 저장되었는지를 확인!

In [28]:
filenames = tf.io.gfile.glob('./data/tfrecord_cat_dog/*.jpg')
print(len(filenames))
print(filenames[0])

4000
./data/tfrecord_cat_dog/cat.620.jpg


In [29]:
# 입력으로 들어온 이미지 1장에 대해 Example 객체를 만들어 리턴하는 함수

def to_example(filename):
    image_string = tf.io.read_file(filename)
    image_shape = tf.io.decode_jpeg(image_string).shape
    label = int((filename.split('/')[-1]).split('.')[0] == 'dog')

    feature = {
        'image/label': tf.train.Feature(int64_list=tf.train.Int64List(value=[label])),
        'image_raw': tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_string.numpy()]))
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))

In [30]:
tfrecord_path = './cat_dog_example.tfrecord'

with tf.io.TFRecordWriter(tfrecord_path) as writer:
    try:
        for i in filenames:
            tf_example = to_example(i)
            writer.write(tf_example.SerializeToString())
    except:
        print('에러발생')

In [38]:
# Dataset 생성

BATCH_SIZE = 64
IMAGE_SIZE = 380

# 이미지 전처리하고 (이미지,라벨) 튜플로 변환하는 함수
def parse_image(proto):
    # 1. Feature 복구
    feature_description = {
        'image/label': tf.io.FixedLenFeature([],tf.int64),
        'image_raw': tf.io.FixedLenFeature([],tf.string)
}
    # 2. 파싱
    parsed_example = tf.io.parse_single_example(proto, feature_description)

    # 3. 이미지 전처리
    image = tf.io.decode_jpeg(parsed_example['image_raw'], channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE,IMAGE_SIZE])
    image = tf.keras.applications.efficientnet.preprocess_input(image)

    label = parsed_example['image/label']
    return image, label

In [40]:
# Dataset 생성
raw_image_dataset = tf.data.TFRecordDataset(tfrecord_path)

parsed_image_dataset = raw_image_dataset.map(parse_image,
                                             num_parallel_calls=tf.data.AUTOTUNE)

# 데이터 수 세기
total_count = len(filenames)

# 재생성
image_dataset = tf.data.TFRecordDataset(tfrecord_path).map(parse_image,
                                             num_parallel_calls=tf.data.AUTOTUNE)

# 학습용 테스트용 나누기
train_size = int(total_count * 0.8)

train_dataset = (image_dataset
                 .shuffle(buffer_size=1000)
                 .take(train_size)
                 .batch(BATCH_SIZE)
                 .prefetch(tf.data.AUTOTUNE))
test_dataset = (image_dataset.skip(train_size)
                .batch(BATCH_SIZE)
                .prefetch(tf.data.AUTOTUNE))

In [41]:
# model
model_base = EfficientNetB4(weights='imagenet',
                            include_top=False,
                            input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
for layer in model_base.layers:
    layer.trainable = False

In [42]:
model = Sequential()
model.add(model_base)
model.add(GlobalAveragePooling2D())
model.add(Dense(units=64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(rate=0.3))
model.add(Dense(units=1,
               activation='sigmoid'))

In [43]:
# model 설정
model.compile(optimizer=Adam(learning_rate=1e-4),
             loss='binary_crossentropy',
             metrics=['accuracy'])

In [44]:
es_callback = EarlyStopping(monitor='val_loss',
                           patience=5,
                           restore_best_weights=True,
                           verbose=1)
cp_callback = ModelCheckpoint(filepath='./cat_dog_tfrecord_weights.h5',
                             save_best_only=True,
                             save_weights_only=True,
                             monitor='val_accuracy',
                             verbose=1)

In [45]:
# 1차 학습 진행
model.fit(train_dataset,
         epochs=30,          
         validation_data=test_dataset,
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/30
     50/Unknown - 27s 298ms/step - loss: 0.3084 - accuracy: 0.8794
Epoch 1: val_accuracy improved from -inf to 0.98375, saving model to ./cat_dog_tfrecord_weights.h5
50/50 [==============================] - 35s 459ms/step - loss: 0.3084 - accuracy: 0.8794 - val_loss: 0.2925 - val_accuracy: 0.9837
Epoch 2/30
50/50 [==============================] - ETA: 0s - loss: 0.0863 - accuracy: 0.9887 
Epoch 2: val_accuracy improved from 0.98375 to 0.99125, saving model to ./cat_dog_tfrecord_weights.h5
50/50 [==============================] - 21s 407ms/step - loss: 0.0863 - accuracy: 0.9887 - val_loss: 0.1745 - val_accuracy: 0.9912
Epoch 3/30
50/50 [==============================] - ETA: 0s - loss: 0.0622 - accuracy: 0.9947 
Epoch 3: val_accuracy improved from 0.99125 to 0.99375, saving model to ./cat_dog_tfrecord_weights.h5
50/50 [==============================] - 20s 405ms/step - loss: 0.0622 - accuracy: 0.9947 - val_loss: 0.1060 - val_accuracy: 0.9937
Epoch 4/30
50/50 [===============

KeyboardInterrupt: 